# 05 - Map-Reduce RAG

**Requires GROQ_API_KEY.**

All previous notebooks retrieved a small number of chunks and passed them
to the LLM. But what if you need to scan an entire document? When the document
is too long to fit in the LLM context window, you need a different strategy.

Map-Reduce RAG processes every chunk independently (MAP) and then combines
the results (REDUCE):

```
Document chunks
  |
  v
MAP: For each chunk, ask the LLM
     "What does this say about the question?"
  |
  v
REDUCE: Combine all answers into one final answer
```

### What you will learn

| Step | Concept |
|------|---------|
| 1 | MAP: extract relevant info from each chunk separately |
| 2 | REDUCE: combine all extracts into one coherent answer |
| 3 | Handle chunks that contain no relevant information |
| 4 | Compare Map-Reduce vs direct retrieval |

> Requires: GROQ_API_KEY in a .env file

## 1. Install and Import

In [ ]:
# !pip install langchain-groq python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()
llm = ChatGroq(model="llama3-8b-8192", temperature=0.2)
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 2. Load and Chunk a Single Document

In [ ]:
DATA_DIR = Path("../data")

def fixed_size_chunks(text, source, chunk_words=120):
    """Split text into non-overlapping fixed-size chunks."""    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_words):
        segment = " ".join(words[i:i + chunk_words])
        if len(segment.split()) >= 30:  # skip very short final chunks
            chunks.append({
                "source": source,
                "chunk_id": len(chunks),
                "text": segment
            })
    return chunks


# Load the climate science document for demonstration
climate_text   = (DATA_DIR / "climate_science.txt").read_text()
climate_chunks = fixed_size_chunks(climate_text, "climate_science.txt")

print(f"Document: climate_science.txt")
print(f"Words   : {len(climate_text.split())}")
print(f"Chunks  : {len(climate_chunks)}")
print()
print("Chunk 0 preview:")
print(" ", climate_chunks[0]["text"][:150])

## 3. MAP Step - Extract from Each Chunk

For every chunk, we ask the LLM: "What does this passage say about the query?"

If the chunk is irrelevant, the LLM responds with "NO_RELEVANT_INFO" so we
can skip it during the reduce step.

In [ ]:
MAP_SYSTEM = (
    "You are an information extraction assistant. "
    "Read the passage below and extract any information relevant to the question. "
    "If the passage contains no relevant information, respond with exactly: NO_RELEVANT_INFO. "
    "Otherwise, write 1-3 concise sentences summarizing what the passage says about the question."
)

def map_chunk(chunk, query):
    """Extract relevant information from one chunk."""    prompt = f"Question: {query}\n\nPassage:\n{chunk['text']}"
    response = llm.invoke([
        SystemMessage(content=MAP_SYSTEM),
        HumanMessage(content=prompt),
    ])
    return response.content.strip()


def map_all_chunks(chunks, query):
    """Run the MAP step over all chunks. Returns only non-empty extracts."""    extracts = []
    print(f"Mapping {len(chunks)} chunks...")
    for i, chunk in enumerate(chunks):
        result = map_chunk(chunk, query)
        status = "relevant" if result != "NO_RELEVANT_INFO" else "skipped"
        print(f"  Chunk {i+1:2d}: {status}")
        if result != "NO_RELEVANT_INFO":
            extracts.append({
                "chunk_id": chunk["chunk_id"],
                "source":   chunk["source"],
                "extract":  result
            })
    print(f"\n{len(extracts)} of {len(chunks)} chunks had relevant information")
    return extracts

print("MAP function ready!")

## 4. REDUCE Step - Combine Extracts

In [ ]:
REDUCE_SYSTEM = (
    "You are a synthesis assistant. "
    "Below are several passages that each contain relevant information about a question. "
    "Combine them into one clear, coherent answer. "
    "Do not repeat information. Keep the answer factual and well-organized."
)

def reduce_extracts(extracts, query):
    """Combine all extracts into a single final answer."""    if not extracts:
        return "No relevant information was found in any of the chunks."

    combined = "\n\n".join(
        f"Extract {i+1}:\n{e['extract']}"
        for i, e in enumerate(extracts)
    )

    response = llm.invoke([
        SystemMessage(content=REDUCE_SYSTEM),
        HumanMessage(content=f"Question: {query}\n\nExtracts:\n{combined}")
    ])
    return response.content.strip()

print("REDUCE function ready!")

## 5. Full Map-Reduce Pipeline

In [ ]:
def map_reduce_rag(query, chunks):
    """Run the full Map-Reduce pipeline."""    print(f"Query: {query}")
    print("=" * 60)

    # MAP
    print("\n[MAP] Extracting relevant information from each chunk...")
    extracts = map_all_chunks(chunks, query)

    # REDUCE
    print("\n[REDUCE] Combining extracts into a final answer...")
    answer = reduce_extracts(extracts, query)

    print(f"\n[FINAL ANSWER]\n{answer}")
    return answer, extracts

_, extracts = map_reduce_rag(
    "What are the main causes and effects of rising sea levels?",
    climate_chunks
)

## 6. Run Across Multiple Documents

In [ ]:
# Load and chunk all four documents
all_chunks = []
for fp in sorted(DATA_DIR.glob("*.txt")):
    chunks = fixed_size_chunks(fp.read_text(), fp.name)
    all_chunks.extend(chunks)
    print(f"  {fp.name:35s}  {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

In [ ]:
# This runs the MAP step over all chunks from all documents
answer, extracts = map_reduce_rag(
    "What were the major contributions of ancient Rome to modern civilization?",
    all_chunks
)

## 7. Inspect the Extracts

You can examine exactly which chunks contributed to the final answer and what
each one said.

In [ ]:
print(f"Extracts used in the answer ({len(extracts)} total):\n")
for i, e in enumerate(extracts, 1):
    print(f"[{i}] Source: {e['source']}  Chunk: {e['chunk_id']}")
    print(f"    Extract: {e['extract'][:150]}...")
    print()

## 8. Map-Reduce vs Direct Retrieval

| Approach | Reads | Good for |
|----------|-------|---------|
| BM25 retrieval | Top-k chunks only | Specific factual questions |
| Reranker RAG | Top-n candidates | Semantic questions |
| Map-Reduce RAG | Every chunk | Comprehensive coverage, long documents |

**Map-Reduce costs more** because it makes one LLM call per chunk plus one
reduce call. For 20 chunks that is 21 LLM calls vs 1 for direct retrieval.

Use Map-Reduce when:
- The document is too long for a single context window
- You need coverage of every section, not just the most relevant parts
- The question asks for a summary or comparison across the whole document

## 9. What You Have Learned Across All Five Notebooks

| Notebook | Technique | LLM calls per query |
|----------|-----------|---------------------|
| 01 | TF-IDF from scratch | 0 |
| 02 | BM25 retrieval | 0 |
| 03 | Sliding window + BM25 + generation | 1 |
| 04 | BM25 + LLM reranking + generation | N candidates + 1 |
| 05 | Map-Reduce RAG | N chunks + 1 |

All five work without a vector database. They rely on keyword statistics and
language model reasoning instead of dense vector similarity.